In [ ]:
from pathlib import Path
from datetime import datetime

# 0) 경로 설정 (여기만 바꾸면 됨)
# ✅ 네 통합 원본 word CSV (한 개)
ORIG_WORD_CSV = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.1.csv")   # <-- 여기 바꿔

# ✅ 네가 검토 후 수정한 파일
CORRECTIONS_CSV = Path(r"..\csv\original_csv\수정파일\(문장확인)바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.1_V_affix_20260112.csv")   # 업로드된 파일이면 그대로 두고, 로컬이면 경로 바꿔

# ✅ 출력 폴더
OUT_DIR = Path(r"../csv/_patched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_CSV = OUT_DIR / f"{ORIG_WORD_CSV.stem}__backup__{stamp}.csv"
PATCHED_CSV = OUT_DIR / f"{ORIG_WORD_CSV.stem}__patched__{stamp}.csv"
AUDIT_CSV  = OUT_DIR / "patch_info"/f"{ORIG_WORD_CSV.stem}__audit__{stamp}.csv"


In [ ]:
import pandas as pd
import numpy as np

# 1) 보조 함수들 (매칭키 자동 선택 + 안전 비교)

IGNORE_COLS = {"Column1", "word_form"}  # 엑셀 보조/문장텍스트 컨텍스트 컬럼은 기본 무시

def safe_equal(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b

def detect_key_mode(corr: pd.DataFrame, orig: pd.DataFrame) -> str:
    """ID가 있으면 ID, 아니면 (file_id, sen_id, word_id2)"""
    if "ID" in corr.columns and "ID" in orig.columns:
        return "ID"
    needed = {"file_id", "sen_id", "word_id2"}
    if needed.issubset(corr.columns) and needed.issubset(orig.columns):
        return "TRIPLE"
    raise KeyError(
        "매칭 키를 못 찾았어. corrections와 orig 둘 다에 "
        "1) ID 가 있거나, 2) file_id+sen_id+word_id2 가 있어야 해."
    )

# 2) 파일 로드 + 키모드 결정 + 백업 저장
orig = pd.read_csv(ORIG_WORD_CSV, encoding="utf-8-sig")
corr = pd.read_csv(CORRECTIONS_CSV, encoding="utf-8-sig")

# correction에서 보조 컬럼 제거
corr = corr.drop(columns=[c for c in IGNORE_COLS if c in corr.columns], errors="ignore")

key_mode = detect_key_mode(corr, orig)
print("key_mode =", key_mode)

# ✅ 원본 백업 저장
orig.to_csv(BACKUP_CSV, index=False, encoding="utf-8-sig")
print("backup ->", BACKUP_CSV)

# 3) 수정 반영 + audit 생성 (핵심)
# 어떤 컬럼을 실제로 수정할지: (corr ∩ orig) - (키 컬럼들)
if key_mode == "ID":
    key_cols = ["ID"]
else:
    key_cols = ["file_id", "sen_id", "word_id2"]

update_cols = [c for c in corr.columns if c in orig.columns and c not in key_cols]
print("update_cols:", update_cols)

audit_rows = []
patched = orig.copy()

if key_mode == "ID":
    # ID는 string으로 통일
    patched["ID"] = patched["ID"].astype("string")
    corr_use = corr.copy()
    corr_use["ID"] = corr_use["ID"].astype("string")

    # correction 중복키 있으면 마지막 값이 이김
    corr_use = corr_use.dropna(subset=["ID"]).drop_duplicates(subset=["ID"], keep="last")

    patched = patched.set_index("ID", drop=False)
    corr_use = corr_use.set_index("ID", drop=False)

    common = patched.index.intersection(corr_use.index)
    print("matched rows:", len(common))

    n_changed = 0
    for cid in common:
        for col in update_cols:
            old = patched.at[cid, col]
            new = corr_use.at[cid, col]
            if safe_equal(old, new):
                continue

            patched.at[cid, col] = new
            n_changed += 1

            audit_rows.append({
                "key_mode": "ID",
                "ID": cid,
                "file_id": patched.at[cid, "file_id"] if "file_id" in patched.columns else pd.NA,
                "sen_id": patched.at[cid, "sen_id"] if "sen_id" in patched.columns else pd.NA,
                "word_id2": patched.at[cid, "word_id2"] if "word_id2" in patched.columns else pd.NA,
                "column": col,
                "old": old,
                "new": new,
            })

    patched = patched.reset_index(drop=True)
    print("changed cells:", n_changed)

else:
    # TRIPLE 키 정규화
    patched["file_id"] = patched["file_id"].astype("string")
    patched["sen_id"]  = patched["sen_id"].astype("string")
    patched["word_id2"] = pd.to_numeric(patched["word_id2"], errors="coerce").astype("Int64")

    corr_use = corr.copy()
    corr_use["file_id"] = corr_use["file_id"].astype("string")
    corr_use["sen_id"]  = corr_use["sen_id"].astype("string")
    corr_use["word_id2"] = pd.to_numeric(corr_use["word_id2"], errors="coerce").astype("Int64")

    corr_use = corr_use.dropna(subset=["file_id","sen_id","word_id2"]).drop_duplicates(
        subset=["file_id","sen_id","word_id2"], keep="last"
    )

    patched = patched.set_index(["file_id","sen_id","word_id2"], drop=False)
    corr_use = corr_use.set_index(["file_id","sen_id","word_id2"], drop=False)

    common = patched.index.intersection(corr_use.index)
    print("matched rows:", len(common))

    n_changed = 0
    for k in common:
        for col in update_cols:
            old = patched.at[k, col]
            new = corr_use.at[k, col]
            if safe_equal(old, new):
                continue

            patched.at[k, col] = new
            n_changed += 1

            audit_rows.append({
                "key_mode": "TRIPLE",
                "ID": patched.at[k, "ID"] if "ID" in patched.columns else pd.NA,
                "file_id": k[0],
                "sen_id": k[1],
                "word_id2": k[2],
                "column": col,
                "old": old,
                "new": new,
            })

    patched = patched.reset_index(drop=True)
    print("changed cells:", n_changed)

audit = pd.DataFrame(audit_rows)
print("audit rows:", len(audit))

# 4) 저장 (패치본 + audit 로그)
patched.to_csv(PATCHED_CSV, index=False, encoding="utf-8-sig")
audit.to_csv(AUDIT_CSV, index=False, encoding="utf-8-sig")

print("patched ->", PATCHED_CSV)
print("audit   ->", AUDIT_CSV)

In [ ]:
# 4) 저장 (패치본 + audit 로그)
patched.to_csv(PATCHED_CSV, index=False, encoding="utf-8-sig")
audit.to_csv(AUDIT_CSV, index=False, encoding="utf-8-sig")

print("patched ->", PATCHED_CSV)
print("audit   ->", AUDIT_CSV)
